# rrf_reciprocal_rank_fusion

# Reciprocal Rank Fusion (RRF)

## What is RRF?

**Reciprocal Rank Fusion** combines results from multiple search methods (e.g., semantic + keyword) into a single ranked list.

**Core Purpose**: Merge rankings from different retrievers to get better results than any single method.

`EnsembleRetriever` - RRF is performed using **EnsembleRetriever**

---

## How It Works

In [ ]:
**Formula**:
RRF_score(doc) = Σ 1 / (k + rank_i(doc))

- k = constant (typically 60)

- rank_i = position of document in retriever i

**Example**:

In [ ]:
Vector Search:           BM25 Keyword:
1. Doc A                 1. Doc C
2. Doc B                 2. Doc A
3. Doc C                 3. Doc D

RRF Scores (k=60):
Doc A: 1/(60+1) + 1/(60+2) = 0.0325 ← appears in both
Doc C: 1/(60+3) + 1/(60+1) = 0.0323
Doc B: 1/(60+2) + 0        = 0.0161
Doc D: 0        + 1/(60+3) = 0.0159

Final Ranking: A → C → B → D

---

## Code Example

In [ ]:
from langchain.retrievers import EnsembleRetriever
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_openai import OpenAIEmbeddings

# 1. Semantic search retriever
vectorstore = Chroma.from_documents(documents, OpenAIEmbeddings())
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# 2. Keyword search retriever
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 5

# 3. Combine with RRF (EnsembleRetriever uses RRF)
ensemble_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.5, 0.5]  # Equal weight
)

# 4. Retrieve with RRF fusion
results = ensemble_retriever.invoke("What is machine learning?")

---

## RRF vs Flash Reranker

**Different purposes:**

| Aspect | RRF | Flash Reranker |

|--------|-----|----------------|

| **Purpose** | Merge multiple retrievers | Score document relevance |

| **Input** | Multiple ranked lists | Query + documents |

| **Method** | Math formula: 1/(k+rank) | ML cross-encoder |

| **Model** | None (algorithmic) | Neural network (120MB) |

| **Speed** | Instant | 8ms |

| **Use Case** | Hybrid search (vector+BM25) | Single retriever refinement |

**Key Difference:**

- **RRF**: Combines rankings (no intelligence, just math)

- **Reranker**: Scores relevance (understands query-doc relationship)

In [ ]:
**Can use both together:**
# Step 1: RRF - Combine vector + BM25 (50 docs)
ensemble_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.5, 0.5]
)

# Step 2: Flash Reranker - Re-score (top 5)
from langchain_community.document_compressors import FlashrankRerank
compressor = FlashrankRerank()
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=ensemble_retriever
)

final_docs = compression_retriever.invoke(query)

In [ ]:
**Pipeline:**
Vector Search (50 docs)  ─┐
                          ├─→ RRF → Flash Reranker → Top 5 Best Docs
BM25 Search (50 docs)    ─┘

---

## When to Use RRF

**Use when:**

- Combining semantic + keyword search (hybrid search)

- Merging multiple retrievers

- Need exact matches + semantic understanding

**Benefit**: 10-20% improvement over single retriever

**Use Cases**: 

- Technical docs (exact terms + semantic meaning)

- Product codes/IDs (keyword) + descriptions (semantic)

- Legal/medical docs (precision + recall)

---

## Key Takeaways

1. **Purpose**: Merge multiple retriever rankings (algorithmic)

2. **vs Reranker**: RRF merges, Reranker scores relevance

3. **Formula**: RRF = Σ 1/(k + rank) across retrievers

4. **Best together**: RRF → Reranker for optimal results

5. **Implementation**: `EnsembleRetriever` in LangChain